# Comprehensive Exploratory Data Analysis (EDA) - Time Series Healthcare Data

This notebook performs detailed EDA on the h5ad time series healthcare dataset.

## Analysis Overview:
- Data loading and basic overview
- Metadata examination (cells and genes)
- Time series analysis
- Data quality assessment
- Statistical analysis and visualizations
- Correlation and dimensionality reduction
- Summary and recommendations

In [1]:
# Import necessary libraries
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization parameters
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")
print(f"Scanpy version: {sc.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Scanpy version: 1.11.5
Pandas version: 2.2.2
NumPy version: 1.26.4


In [ ]:
# Load the h5ad data file
print("Loading h5ad data file...")

# The h5ad file is located in the Degree directory
file_path = 'h5ad.h5ad'

adata = sc.read_h5ad(file_path)
print(f"Data loaded successfully from: {file_path}")
print(f"Data shape: {adata.shape}")
print(f"AnnData object created: {type(adata)}")

Loading h5ad data file...
File not found at: ss2 (1)/ss2/adata.ct.h5ad
Trying alternative path...
File not found at: ss2/ss2/h5ad.h5ad
Please check the correct path to your h5ad file


In [ ]:
# Basic Data Overview
print("=== BASIC DATA OVERVIEW ===")
print(f"Number of cells (observations): {adata.n_obs:,}")
print(f"Number of genes (features): {adata.n_vars:,}")
print(f"Data type: {type(adata)}")
print(f"AnnData object keys: {list(adata.keys()) if hasattr(adata, 'keys') else 'No keys'}")

# Check if layers exist
if hasattr(adata, 'layers') and adata.layers:
    print(f"\nAvailable layers: {list(adata.layers.keys())}")
else:
    print("\nNo additional layers found.")

# Check for uns (unstructured annotations)
if hasattr(adata, 'uns') and adata.uns:
    print(f"Unstructured annotations keys: {list(adata.uns.keys())}")
else:
    print("No unstructured annotations found.")

In [ ]:
# Examine observation metadata (cell-level information)
print("=== OBSERVATION METADATA (CELL-LEVEL) ===")
if adata.obs is not None and not adata.obs.empty:
    print(f"Observation metadata shape: {adata.obs.shape}")
    print("\nColumn names:")
    for i, col in enumerate(adata.obs.columns):
        print(f"  {i+1}. {col}")
    
    print("\nFirst 5 rows of observation metadata:")
    display(adata.obs.head())
    
    print("\nData types of observation metadata:")
    print(adata.obs.dtypes)
    
    print("\nBasic statistics for numeric columns:")
    numeric_cols = adata.obs.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        display(adata.obs[numeric_cols].describe())
    else:
        print("No numeric columns found in observation metadata.")
else:
    print("No observation metadata found.")

In [ ]:
# Examine variable metadata (gene-level information)
print("=== VARIABLE METADATA (GENE-LEVEL) ===")
if adata.var is not None and not adata.var.empty:
    print(f"Variable metadata shape: {adata.var.shape}")
    print("\nColumn names:")
    for i, col in enumerate(adata.var.columns):
        print(f"  {i+1}. {col}")
    
    print("\nFirst 5 rows of variable metadata:")
    display(adata.var.head())
    
    print("\nData types of variable metadata:")
    print(adata.var.dtypes)
    
    # Show gene names if available
    if adata.var_names is not None and len(adata.var_names) > 0:
        print(f"\nTotal gene names: {len(adata.var_names)}")
        print(f"First 10 gene names: {list(adata.var_names[:10])}")
        if len(adata.var_names) > 10:
            print(f"Last 10 gene names: {list(adata.var_names[-10:])}")
else:
    print("No variable metadata found.")

In [ ]:
# Check for time series information
print("=== TIME SERIES INFORMATION DETECTION ===")
time_columns = []
potential_time_keywords = ['time', 'date', 'hour', 'day', 'month', 'year', 'timestamp', 
                          'period', 'stage', 'phase', 'visit', 'session', 'sample_time']

if adata.obs is not None:
    print("Scanning observation metadata for time-related columns...")
    for col in adata.obs.columns:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in potential_time_keywords):
            time_columns.append(col)
    
    if time_columns:
        print(f"\n✓ Potential time-related columns found: {time_columns}")
        for col in time_columns:
            print(f"\n{col}:")
            print(f"  - Unique values: {adata.obs[col].nunique()}")
            print(f"  - Data type: {adata.obs[col].dtype}")
            unique_vals = adata.obs[col].dropna().unique()
            if len(unique_vals) <= 10:
                print(f"  - Values: {sorted(unique_vals)}")
            else:
                print(f"  - Sample values: {sorted(unique_vals)[:5]} ... {sorted(unique_vals)[-5:]}")
    else:
        print("\n✗ No obvious time-related columns found in observation metadata.")
        print("\nAll column names for manual inspection:")
        for i, col in enumerate(adata.obs.columns):
            print(f"  {i+1}. {col}")
else:
    print("No observation metadata to check for time information.")

# Also check var metadata for time-related gene information
if adata.var is not None:
    gene_time_columns = []
    for col in adata.var.columns:
        col_lower = col.lower()
        if any(keyword in col_lower for keyword in potential_time_keywords):
            gene_time_columns.append(col)
    
    if gene_time_columns:
        print(f"\n✓ Time-related columns in gene metadata: {gene_time_columns}")

In [ ]:
# Examine the main data matrix
print("=== DATA MATRIX ANALYSIS ===")
print(f"Data matrix shape: {adata.X.shape}")
print(f"Data matrix type: {type(adata.X)}")

# Check if data is sparse or dense
if hasattr(adata.X, 'format'):
    print(f"\nSparse matrix format: {adata.X.format}")
    sparsity = adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
    print(f"Sparsity: {sparsity:.4f} ({sparsity*100:.2f}% zero)")
    print(f"Non-zero elements: {adata.X.nnz:,}")
    print(f"Total elements: {adata.X.shape[0] * adata.X.shape[1]:,}")
else:
    print("\nData is stored as dense matrix")
    total_elements = adata.X.size
    zero_elements = np.sum(adata.X == 0)
    sparsity = zero_elements / total_elements
    print(f"Sparsity: {sparsity:.4f} ({sparsity*100:.2f}% zero)")
    print(f"Zero elements: {zero_elements:,}")
    print(f"Total elements: {total_elements:,}")

# Basic statistics
if hasattr(adata.X, 'data'):
    # For sparse matrices
    data_values = adata.X.data
    print(f"\nData statistics (non-zero values only):")
else:
    # For dense matrices
    data_values = adata.X.flatten()
    print(f"\nData statistics (all values):")

print(f"  Min value: {np.min(data_values)}")
print(f"  Max value: {np.max(data_values)}")
print(f"  Mean value: {np.mean(data_values):.4f}")
print(f"  Median value: {np.median(data_values):.4f}")
print(f"  Standard deviation: {np.std(data_values):.4f}")
print(f"  25th percentile: {np.percentile(data_values, 25):.4f}")
print(f"  75th percentile: {np.percentile(data_values, 75):.4f}")

In [ ]:
# Data Quality Assessment
print("=== DATA QUALITY ASSESSMENT ===")

# Check for missing values in data matrix
if hasattr(adata.X, 'data'):
    # Sparse matrix - check for explicit zeros vs missing
    total_elements = adata.X.shape[0] * adata.X.shape[1]
    zero_elements = total_elements - adata.X.nnz
    print(f"Data Matrix Quality:")
    print(f"  - Total elements: {total_elements:,}")
    print(f"  - Non-zero elements: {adata.X.nnz:,}")
    print(f"  - Zero elements: {zero_elements:,}")
    print(f"  - Zero percentage: {zero_elements/total_elements*100:.2f}%")
else:
    # Dense matrix
    missing_count = np.sum(np.isnan(adata.X))
    zero_count = np.sum(adata.X == 0)
    print(f"Data Matrix Quality:")
    print(f"  - Missing values (NaN): {missing_count:,}")
    print(f"  - Zero values: {zero_count:,}")
    print(f"  - Missing percentage: {missing_count/adata.X.size*100:.2f}%")
    print(f"  - Zero percentage: {zero_count/adata.X.size*100:.2f}%")

# Check observation metadata for missing values
if adata.obs is not None:
    print(f"\nObservation Metadata Quality:")
    missing_obs = adata.obs.isnull().sum()
    if missing_obs.sum() > 0:
        print("  - Missing values by column:")
        for col, count in missing_obs[missing_obs > 0].items():
            print(f"    {col}: {count} ({count/len(adata.obs)*100:.2f}%)")
    else:
        print("  - No missing values found in observation metadata")

# Check variable metadata for missing values
if adata.var is not None:
    print(f"\nVariable Metadata Quality:")
    missing_var = adata.var.isnull().sum()
    if missing_var.sum() > 0:
        print("  - Missing values by column:")
        for col, count in missing_var[missing_var > 0].items():
            print(f"    {col}: {count} ({count/len(adata.var)*100:.2f}%)")
    else:
        print("  - No missing values found in variable metadata")

In [ ]:
# Cell-level Statistics
print("=== CELL-LEVEL STATISTICS ===")

# Calculate basic statistics per cell
if hasattr(adata.X, 'data'):
    # For sparse matrices
    cell_counts = np.array(adata.X.sum(axis=1)).flatten()
    cell_genes = np.array((adata.X > 0).sum(axis=1)).flatten()
else:
    # For dense matrices
    cell_counts = np.sum(adata.X, axis=1)
    cell_genes = np.sum(adata.X > 0, axis=1)

print(f"Total counts per cell:")
print(f"  Min: {np.min(cell_counts):,}")
print(f"  Max: {np.max(cell_counts):,}")
print(f"  Mean: {np.mean(cell_counts):.2f}")
print(f"  Median: {np.median(cell_counts):.2f}")
print(f"  Std: {np.std(cell_counts):.2f}")
print(f"  25th percentile: {np.percentile(cell_counts, 25):.2f}")
print(f"  75th percentile: {np.percentile(cell_counts, 75):.2f}")

print(f"\nGenes detected per cell:")
print(f"  Min: {np.min(cell_genes):,}")
print(f"  Max: {np.max(cell_genes):,}")
print(f"  Mean: {np.mean(cell_genes):.2f}")
print(f"  Median: {np.median(cell_genes):.2f}")
print(f"  Std: {np.std(cell_genes):.2f}")
print(f"  25th percentile: {np.percentile(cell_genes, 25):.2f}")
print(f"  75th percentile: {np.percentile(cell_genes, 75):.2f}")

# Identify potential outliers
q1_counts, q3_counts = np.percentile(cell_counts, [25, 75])
iqr_counts = q3_counts - q1_counts
lower_bound_counts = q1_counts - 1.5 * iqr_counts
upper_bound_counts = q3_counts + 1.5 * iqr_counts
outlier_cells = np.sum((cell_counts < lower_bound_counts) | (cell_counts > upper_bound_counts))
print(f"\nOutlier Analysis (based on total counts):")
print(f"  - Cells identified as outliers: {outlier_cells} ({outlier_cells/len(cell_counts)*100:.2f}%)")
print(f"  - Lower bound: {lower_bound_counts:.2f}")
print(f"  - Upper bound: {upper_bound_counts:.2f}")

In [ ]:
# Visualize cell-level statistics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Total counts per cell histogram
axes[0, 0].hist(cell_counts, bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].set_title('Distribution of Total Counts per Cell')
axes[0, 0].set_xlabel('Total Counts')
axes[0, 0].set_ylabel('Number of Cells')
axes[0, 0].axvline(np.mean(cell_counts), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(cell_counts):.0f}')
axes[0, 0].axvline(np.median(cell_counts), color='orange', linestyle='--', 
                   label=f'Median: {np.median(cell_counts):.0f}')
axes[0, 0].legend()

# Genes detected per cell histogram
axes[0, 1].hist(cell_genes, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_title('Distribution of Genes Detected per Cell')
axes[0, 1].set_xlabel('Number of Genes')
axes[0, 1].set_ylabel('Number of Cells')
axes[0, 1].axvline(np.mean(cell_genes), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(cell_genes):.0f}')
axes[0, 1].axvline(np.median(cell_genes), color='orange', linestyle='--', 
                   label=f'Median: {np.median(cell_genes):.0f}')
axes[0, 1].legend()

# Scatter plot of counts vs genes
axes[0, 2].scatter(cell_counts, cell_genes, alpha=0.5, s=1)
axes[0, 2].set_title('Total Counts vs Genes Detected per Cell')
axes[0, 2].set_xlabel('Total Counts')
axes[0, 2].set_ylabel('Number of Genes')

# Box plot
axes[1, 0].boxplot([cell_counts, cell_genes], labels=['Total Counts', 'Genes Detected'])
axes[1, 0].set_title('Box Plot of Cell Statistics')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=45)

# Violin plot
parts = axes[1, 1].violinplot([cell_counts, cell_genes], 
                              positions=[1, 2], showmeans=True)
axes[1, 1].set_title('Violin Plot of Cell Statistics')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_xticks([1, 2])
axes[1, 1].set_xticklabels(['Total Counts', 'Genes Detected'])

# Cumulative distribution
sorted_counts = np.sort(cell_counts)
cumulative = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
axes[1, 2].plot(sorted_counts, cumulative)
axes[1, 2].set_title('Cumulative Distribution of Total Counts')
axes[1, 2].set_xlabel('Total Counts')
axes[1, 2].set_ylabel('Cumulative Proportion')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Gene-level Statistics
print("=== GENE-LEVEL STATISTICS ===")

# Calculate basic statistics per gene
if hasattr(adata.X, 'data'):
    # For sparse matrices
    gene_counts = np.array(adata.X.sum(axis=0)).flatten()
    gene_cells = np.array((adata.X > 0).sum(axis=0)).flatten()
else:
    # For dense matrices
    gene_counts = np.sum(adata.X, axis=0)
    gene_cells = np.sum(adata.X > 0, axis=0)

print(f"Total counts per gene:")
print(f"  Min: {np.min(gene_counts):,}")
print(f"  Max: {np.max(gene_counts):,}")
print(f"  Mean: {np.mean(gene_counts):.2f}")
print(f"  Median: {np.median(gene_counts):.2f}")
print(f"  Std: {np.std(gene_counts):.2f}")

print(f"\nCells expressing each gene:")
print(f"  Min: {np.min(gene_cells):,}")
print(f"  Max: {np.max(gene_cells):,}")
print(f"  Mean: {np.mean(gene_cells):.2f}")
print(f"  Median: {np.median(gene_cells):.2f}")
print(f"  Std: {np.std(gene_cells):.2f}")

# Gene expression metrics
zero_genes = np.sum(gene_counts == 0)
low_expressed_genes = np.sum(gene_cells < 0.01 * adata.n_obs)  # Expressed in < 1% of cells
highly_expressed_genes = np.sum(gene_cells > 0.5 * adata.n_obs)   # Expressed in > 50% of cells

print(f"\nGene Expression Summary:")
print(f"  - Genes with zero counts: {zero_genes:,} ({zero_genes/len(gene_counts)*100:.2f}%)")
print(f"  - Genes expressed in < 1% of cells: {low_expressed_genes:,} ({low_expressed_genes/len(gene_counts)*100:.2f}%)")
print(f"  - Genes expressed in > 50% of cells: {highly_expressed_genes:,} ({highly_expressed_genes/len(gene_counts)*100:.2f}%)")

# Highly expressed genes
top_genes_idx = np.argsort(gene_counts)[-10:][::-1]
if adata.var_names is not None and len(adata.var_names) > 0:
    top_genes = adata.var_names[top_genes_idx]
    print(f"\nTop 10 highly expressed genes:")
    for i, (gene, count) in enumerate(zip(top_genes, gene_counts[top_genes_idx])):
        cells_expressing = gene_cells[top_genes_idx[i]]
        percentage = cells_expressing / adata.n_obs * 100
        print(f"  {i+1}. {gene}: {count:,} counts (expressed in {cells_expressing:,} cells, {percentage:.1f}%)")

In [ ]:
# Visualize gene-level statistics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Total counts per gene histogram (non-zero only)
non_zero_gene_counts = gene_counts[gene_counts > 0]
axes[0, 0].hist(non_zero_gene_counts, bins=50, alpha=0.7, color='purple', edgecolor='black')
axes[0, 0].set_title('Distribution of Total Counts per Gene (non-zero)')
axes[0, 0].set_xlabel('Total Counts')
axes[0, 0].set_ylabel('Number of Genes')
axes[0, 0].set_yscale('log')
axes[0, 0].set_xscale('log')

# Cells expressing each gene histogram (non-zero only)
non_zero_gene_cells = gene_cells[gene_cells > 0]
axes[0, 1].hist(non_zero_gene_cells, bins=50, alpha=0.7, color='orange', edgecolor='black')
axes[0, 1].set_title('Distribution of Cells per Gene (non-zero)')
axes[0, 1].set_xlabel('Number of Cells')
axes[0, 1].set_ylabel('Number of Genes')
axes[0, 1].set_yscale('log')

# Scatter plot of counts vs cells
non_zero_mask = (gene_counts > 0) & (gene_cells > 0)
axes[0, 2].scatter(gene_counts[non_zero_mask], gene_cells[non_zero_mask], alpha=0.5, s=1)
axes[0, 2].set_title('Total Counts vs Cells per Gene')
axes[0, 2].set_xlabel('Total Counts')
axes[0, 2].set_ylabel('Number of Cells')
axes[0, 2].set_xscale('log')
axes[0, 2].set_yscale('log')

# Top expressed genes bar plot
if adata.var_names is not None and len(adata.var_names) > 0:
    top_20_idx = np.argsort(gene_counts)[-20:][::-1]
    top_20_genes = adata.var_names[top_20_idx]
    top_20_counts = gene_counts[top_20_idx]
    
    y_pos = np.arange(len(top_20_genes))
    axes[1, 0].barh(y_pos, top_20_counts, color='skyblue')
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels(top_20_genes, fontsize=8)
    axes[1, 0].invert_yaxis()
    axes[1, 0].set_title('Top 20 Expressed Genes')
    axes[1, 0].set_xlabel('Total Counts')

# Gene detection distribution
axes[1, 1].hist(gene_cells / adata.n_obs * 100, bins=50, alpha=0.7, color='red', edgecolor='black')
axes[1, 1].set_title('Distribution of Gene Detection Rate')
axes[1, 1].set_xlabel('Percentage of Cells Expressing Gene (%)')
axes[1, 1].set_ylabel('Number of Genes')
axes[1, 1].axvline(1, color='blue', linestyle='--', label='1% threshold')
axes[1, 1].axvline(50, color='green', linestyle='--', label='50% threshold')
axes[1, 1].legend()

# Cumulative gene expression
sorted_gene_counts = np.sort(gene_counts[gene_counts > 0])
cumulative_genes = np.arange(1, len(sorted_gene_counts) + 1) / len(gene_counts)
axes[1, 2].plot(sorted_gene_counts, cumulative_genes)
axes[1, 2].set_title('Cumulative Distribution of Gene Expression')
axes[1, 2].set_xlabel('Total Counts')
axes[1, 2].set_ylabel('Cumulative Proportion of Genes')
axes[1, 2].set_xscale('log')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Time Series Analysis (if time information is available)
print("=== TIME SERIES ANALYSIS ===")

if time_columns:
    for time_col in time_columns:
        print(f"\n--- Analyzing time column: {time_col} ---")
        
        # Get unique time points
        time_points = adata.obs[time_col].dropna().unique()
        print(f"Number of unique time points: {len(time_points)}")
        
        if len(time_points) <= 20:
            print(f"Time points: {sorted(time_points)}")
        else:
            print(f"Time points range: {np.min(time_points)} to {np.max(time_points)}")
        
        # Count cells per time point
        cells_per_time = adata.obs[time_col].value_counts().sort_index()
        print(f"\nCells per time point (first 10):")
        print(cells_per_time.head(10))
        
        # Visualize cells over time
        plt.figure(figsize=(14, 6))
        
        if len(time_points) <= 50:
            plt.subplot(1, 2, 1)
            plt.bar(range(len(cells_per_time)), cells_per_time.values, alpha=0.7)
            plt.xticks(range(len(cells_per_time)), cells_per_time.index, rotation=45)
            plt.xlabel('Time Point')
            plt.ylabel('Number of Cells')
            plt.title(f'Cells per Time Point ({time_col})')
        else:
            plt.subplot(1, 2, 1)
            plt.plot(cells_per_time.index, cells_per_time.values, marker='o', markersize=3, alpha=0.7)
            plt.xlabel('Time')
            plt.ylabel('Number of Cells')
            plt.title(f'Cells Over Time ({time_col})')
        
        # Cell statistics over time
        plt.subplot(1, 2, 2)
        time_stats = []
        time_labels = []
        
        for tp in sorted(time_points):
            mask = adata.obs[time_col] == tp
            if mask.sum() > 0:
                if hasattr(adata.X, 'data'):
                    tp_counts = np.array(adata.X[mask].sum(axis=1)).flatten()
                else:
                    tp_counts = np.sum(adata.X[mask], axis=1)
                time_stats.append(tp_counts)
                time_labels.append(str(tp))
        
        if len(time_stats) > 1:
            plt.boxplot(time_stats, labels=time_labels)
            plt.xlabel('Time Point')
            plt.ylabel('Total Counts per Cell')
            plt.title('Cell Count Distribution Over Time')
            plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # Analyze changes in gene expression over time
        if len(time_points) > 1 and len(time_points) <= 10:
            print(f"\nAnalyzing gene expression changes over time...")
            
            # Calculate average expression per time point
            time_expr = []
            
            for tp in sorted(time_points):
                mask = adata.obs[time_col] == tp
                if mask.sum() > 0:
                    avg_expr = np.mean(adata.X[mask].toarray() if hasattr(adata.X, 'toarray') else adata.X[mask], axis=0)
                    time_expr.append(avg_expr)
            
            if len(time_expr) > 1:
                # Find most variable genes over time
                time_expr_array = np.array(time_expr)
                gene_var = np.var(time_expr_array, axis=0)
                top_var_genes_idx = np.argsort(gene_var)[-5:][::-1]
                
                if adata.var_names is not None and len(adata.var_names) > 0:
                    top_var_genes = adata.var_names[top_var_genes_idx]
                    
                    # Plot expression of top variable genes over time
                    plt.figure(figsize=(12, 8))
                    for i, gene in enumerate(top_var_genes):
                        plt.plot(sorted(time_points), time_expr_array[:, top_var_genes_idx[i]], 
                                marker='o', label=gene, linewidth=2, markersize=6)
                    
                    plt.xlabel('Time Point')
                    plt.ylabel('Average Expression')
                    plt.title('Most Variable Genes Over Time')
                    plt.legend()
                    plt.xticks(rotation=45)
                    plt.grid(True, alpha=0.3)
                    plt.tight_layout()
                    plt.show()
                    
                    print(f"\nTop 5 most variable genes over time:")
                    for i, gene in enumerate(top_var_genes):
                        print(f"  {i+1}. {gene}: variance = {gene_var[top_var_genes_idx[i]]:.4f}")
else:
    print("No time-related columns found for time series analysis.")
    print("Consider checking if any columns contain temporal information that wasn't automatically detected.")

In [ ]:
# Categorical Variable Analysis
print("=== CATEGORICAL VARIABLE ANALYSIS ===")

if adata.obs is not None:
    categorical_columns = adata.obs.select_dtypes(include=['object', 'category']).columns
    
    # Filter out time columns that were already analyzed
    categorical_columns = [col for col in categorical_columns if col not in time_columns]
    
    if len(categorical_columns) > 0:
        print(f"Found {len(categorical_columns)} non-time categorical columns:")
        
        for col in categorical_columns:
            print(f"\n--- Analyzing column: {col} ---")
            value_counts = adata.obs[col].value_counts()
            print(f"Number of unique values: {len(value_counts)}")
            print(f"Value counts:")
            print(value_counts)
            
            # Calculate percentages
            percentages = (value_counts / len(adata.obs) * 100).round(2)
            print(f"\nPercentages:")
            for val, pct in percentages.items():
                print(f"  {val}: {pct}%")
            
            # Visualize distribution
            if len(value_counts) <= 20:
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
                
                # Bar plot
                value_counts.plot(kind='bar', ax=ax1, alpha=0.7)
                ax1.set_title(f'Distribution of {col}')
                ax1.set_xlabel(col)
                ax1.set_ylabel('Count')
                ax1.tick_params(axis='x', rotation=45)
                
                # Pie chart
                ax2.pie(value_counts.values, labels=value_counts.index, autopct='%1.1f%%', startangle=90)
                ax2.set_title(f'Proportion of {col}')
                
                plt.tight_layout()
                plt.show()
            else:
                print(f"Skipping visualization for {col} (too many categories: {len(value_counts)})")
    else:
        print("No non-time categorical columns found in observation metadata.")
else:
    print("No observation metadata available for categorical analysis.")

# Also check for numeric columns that might be categorical
if adata.obs is not None:
    numeric_cols = adata.obs.select_dtypes(include=[np.number]).columns
    potential_categorical_numeric = []
    
    for col in numeric_cols:
        if col not in time_columns:
            unique_vals = adata.obs[col].nunique()
            if unique_vals <= 10 and unique_vals < len(adata.obs) * 0.1:
                potential_categorical_numeric.append(col)
    
    if potential_categorical_numeric:
        print(f"\nNumeric columns that might be categorical (≤10 unique values):")
        for col in potential_categorical_numeric:
            print(f"  - {col}: {adata.obs[col].nunique()} unique values")

In [ ]:
# Correlation Analysis
print("=== CORRELATION ANALYSIS ===")

# Sample cells for correlation analysis (to avoid memory issues)
sample_size = min(1000, adata.n_obs)
print(f"Sampling {sample_size} cells for correlation analysis...")

np.random.seed(42)  # For reproducibility
sample_indices = np.random.choice(adata.n_obs, sample_size, replace=False)

if hasattr(adata.X, 'toarray'):
    sample_data = adata.X[sample_indices].toarray()
else:
    sample_data = adata.X[sample_indices]

print(f"Sample data shape: {sample_data.shape}")

# Calculate correlation matrix for highly expressed genes
gene_means = np.mean(sample_data, axis=0)
top_genes_idx = np.argsort(gene_means)[-100:]  # Top 100 expressed genes
sample_data_top = sample_data[:, top_genes_idx]

print(f"Analyzing top 100 most expressed genes...")
correlation_matrix = np.corrcoef(sample_data_top.T)

print(f"Correlation matrix shape: {correlation_matrix.shape}")
mask = correlation_matrix != 1
print(f"Mean correlation (excluding self-correlations): {np.mean(correlation_matrix[mask]):.4f}")
print(f"Std correlation (excluding self-correlations): {np.std(correlation_matrix[mask]):.4f}")
print(f"Min correlation: {np.min(correlation_matrix[mask]):.4f}")
print(f"Max correlation: {np.min(correlation_matrix[mask]):.4f}")

# Find highly correlated gene pairs
high_corr_threshold = 0.8
high_corr_pairs = []
for i in range(len(top_genes_idx)):
    for j in range(i+1, len(top_genes_idx)):
        if abs(correlation_matrix[i, j]) > high_corr_threshold:
            high_corr_pairs.append((i, j, correlation_matrix[i, j]))

print(f"\nHighly correlated gene pairs (|r| > {high_corr_threshold}): {len(high_corr_pairs)}")
if high_corr_pairs and adata.var_names is not None:
    print("Top 10 highly correlated pairs:")
    high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    for i, (idx1, idx2, corr) in enumerate(high_corr_pairs[:10]):
        gene1 = adata.var_names[top_genes_idx[idx1]]
        gene2 = adata.var_names[top_genes_idx[idx2]]
        print(f"  {i+1}. {gene1} - {gene2}: r = {corr:.4f}")

# Visualize correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            xticklabels=False, yticklabels=False, 
            vmin=-1, vmax=1, square=True)
plt.title('Gene-Gene Correlation Matrix (Top 100 Expressed Genes)')
plt.tight_layout()
plt.show()

In [ ]:
# Principal Component Analysis
print("=== PRINCIPAL COMPONENT ANALYSIS ===")

# Use the same sample for PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print(f"Performing PCA on {sample_data_top.shape[0]} cells × {sample_data_top.shape[1]} genes...")

# Standardize the data
scaler = StandardScaler(with_mean=False)  # with_mean=False for sparse data
scaled_data = scaler.fit_transform(sample_data_top)

# Perform PCA
n_components = min(10, sample_data_top.shape[1])
pca = PCA(n_components=n_components)
pca_result = pca.fit_transform(scaled_data.toarray() if hasattr(scaled_data, 'toarray') else scaled_data)

print(f"\nPCA Results:")
print(f"Explained variance ratio:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ratio:.4f} ({ratio*100:.2f}%)")

print(f"\nCumulative explained variance:")
cumulative_var = np.cumsum(pca.explained_variance_ratio_)
for i, ratio in enumerate(cumulative_var):
    print(f"  PC{i+1}: {ratio:.4f} ({ratio*100:.2f}%)")

# Find components needed for 80% and 90% variance
pc_80 = np.where(cumulative_var >= 0.8)[0]
pc_90 = np.where(cumulative_var >= 0.9)[0]
if len(pc_80) > 0:
    print(f"\nComponents needed for 80% variance: {pc_80[0] + 1}")
if len(pc_90) > 0:
    print(f"Components needed for 90% variance: {pc_90[0] + 1}")

# Visualize PCA results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Scree plot
axes[0, 0].plot(range(1, len(pca.explained_variance_ratio_) + 1), 
             pca.explained_variance_ratio_, 'bo-', markersize=6)
axes[0, 0].set_xlabel('Principal Component')
axes[0, 0].set_ylabel('Explained Variance Ratio')
axes[0, 0].set_title('Scree Plot')
axes[0, 0].grid(True, alpha=0.3)

# Cumulative variance plot
axes[0, 1].plot(range(1, len(cumulative_var) + 1), cumulative_var, 'ro-', markersize=6)
axes[0, 1].set_xlabel('Principal Component')
axes[0, 1].set_ylabel('Cumulative Explained Variance Ratio')
axes[0, 1].set_title('Cumulative Explained Variance')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=0.8, color='g', linestyle='--', label='80%')
axes[0, 1].axhline(y=0.9, color='orange', linestyle='--', label='90%')
axes[0, 1].legend()

# PC1 vs PC2 scatter plot
axes[1, 0].scatter(pca_result[:, 0], pca_result[:, 1], alpha=0.6, s=20)
axes[1, 0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1, 0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[1, 0].set_title('PC1 vs PC2')
axes[1, 0].grid(True, alpha=0.3)

# Loadings for PC1
if adata.var_names is not None:
    loadings_pc1 = pca.components_[0]
    top_loading_idx = np.argsort(np.abs(loadings_pc1))[-10:][::-1]
    top_genes = adata.var_names[top_genes_idx[top_loading_idx]]
    top_loadings = loadings_pc1[top_loading_idx]
    
    y_pos = np.arange(len(top_genes))
    axes[1, 1].barh(y_pos, top_loadings, color='lightcoral')
    axes[1, 1].set_yticks(y_pos)
    axes[1, 1].set_yticklabels(top_genes, fontsize=8)
    axes[1, 1].invert_yaxis()
    axes[1, 1].set_title('Top Gene Loadings for PC1')
    axes[1, 1].set_xlabel('Loading Value')
    axes[1, 1].axvline(x=0, color='black', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Advanced Analysis: Differential Expression by Categories (if available)
print("=== DIFFERENTIAL EXPRESSION ANALYSIS ===")

# Check if we have categorical variables suitable for DE analysis
if adata.obs is not None:
    categorical_columns = adata.obs.select_dtypes(include=['object', 'category']).columns
    # Filter out time columns and columns with too many categories
    suitable_cols = []
    for col in categorical_columns:
        if col not in time_columns:
            n_unique = adata.obs[col].nunique()
            if 2 <= n_unique <= 5:  # Suitable for DE analysis
                suitable_cols.append(col)
    
    if suitable_cols:
        for col in suitable_cols:
            print(f"\n--- Differential Expression by {col} ---")
            
            # Get groups
            groups = adata.obs[col].value_counts()
            print(f"Groups: {dict(groups)}")
            
            if len(groups) == 2:  # Simple two-group comparison
                group1, group2 = groups.index[:2]
                mask1 = adata.obs[col] == group1
                mask2 = adata.obs[col] == group2
                
                if mask1.sum() > 0 and mask2.sum() > 0:
                    # Calculate mean expression for each group
                    if hasattr(adata.X, 'toarray'):
                        expr1 = adata.X[mask1].toarray()
                        expr2 = adata.X[mask2].toarray()
                    else:
                        expr1 = adata.X[mask1]
                        expr2 = adata.X[mask2]
                    
                    mean1 = np.mean(expr1, axis=0)
                    mean2 = np.mean(expr2, axis=0)
                    
                    # Calculate fold change
                    # Add small constant to avoid division by zero
                    fc = (mean1 + 1e-9) / (mean2 + 1e-9)
                    log2_fc = np.log2(fc)
                    
                    # Find top differentially expressed genes
                    top_de_idx = np.argsort(np.abs(log2_fc))[-10:][::-1]
                    
                    if adata.var_names is not None:
                        top_de_genes = adata.var_names[top_de_idx]
                        top_log2_fc = log2_fc[top_de_idx]
                        
                        print(f"\nTop 10 differentially expressed genes ({group1} vs {group2}):")
                        for i, (gene, lfc) in enumerate(zip(top_de_genes, top_log2_fc)):
                            direction = "up" if lfc > 0 else "down"
                            print(f"  {i+1}. {gene}: log2FC = {lfc:.3f} ({direction} in {group1})")
                        
                        # Visualize
                        plt.figure(figsize=(10, 6))
                        colors = ['red' if lfc > 0 else 'blue' for lfc in top_log2_fc]
                        plt.barh(range(len(top_de_genes)), top_log2_fc, color=colors, alpha=0.7)
                        plt.yticks(range(len(top_de_genes)), top_de_genes)
                        plt.xlabel('Log2 Fold Change')
                        plt.title(f'Top DE Genes: {group1} vs {group2}')
                        plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
                        plt.tight_layout()
                        plt.show()
    else:
        print("No suitable categorical variables found for differential expression analysis.")
        print("(Looking for categorical variables with 2-5 unique values, excluding time columns)")
else:
    print("No observation metadata available for differential expression analysis.")

In [ ]:
# Summary and Key Insights
print("=== COMPREHENSIVE SUMMARY AND KEY INSIGHTS ===")

# Dataset Overview
print("📊 DATASET OVERVIEW:")
print(f"  • Total cells: {adata.n_obs:,}")
print(f"  • Total genes: {adata.n_vars:,}")
print(f"  • Data matrix type: {type(adata.X).__name__}")
if hasattr(adata.X, 'format'):
    print(f"  • Sparse format: {adata.X.format}")
    sparsity = adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
    print(f"  • Sparsity: {sparsity:.4f} ({sparsity*100:.2f}% zero)")

# Cell Statistics
print(f"\n🧬 CELL STATISTICS:")
print(f"  • Average counts per cell: {np.mean(cell_counts):.2f}")
print(f"  • Median counts per cell: {np.median(cell_counts):.2f}")
print(f"  • Average genes per cell: {np.mean(cell_genes):.2f}")
print(f"  • Median genes per cell: {np.median(cell_genes):.2f}")
print(f"  • Cell count range: {np.min(cell_counts):,} - {np.max(cell_counts):,}")
print(f"  • Outlier cells: {outlier_cells} ({outlier_cells/len(cell_counts)*100:.2f}%)")

# Gene Statistics
print(f"\n🧪 GENE STATISTICS:")
print(f"  • Average counts per gene: {np.mean(gene_counts):.2f}")
print(f"  • Median counts per gene: {np.median(gene_counts):.2f}")
print(f"  • Average cells per gene: {np.mean(gene_cells):.2f}")
print(f"  • Median cells per gene: {np.median(gene_cells):.2f}")
print(f"  • Genes with zero counts: {zero_genes:,} ({zero_genes/len(gene_counts)*100:.2f}%)")
print(f"  • Genes expressed in < 1% of cells: {low_expressed_genes:,} ({low_expressed_genes/len(gene_counts)*100:.2f}%)")
print(f"  • Genes expressed in > 50% of cells: {highly_expressed_genes:,} ({highly_expressed_genes/len(gene_counts)*100:.2f}%)")

# Time Series Information
if time_columns:
    print(f"\n⏰ TIME SERIES INFORMATION:")
    print(f"  • Time columns found: {time_columns}")
    for col in time_columns:
        unique_times = adata.obs[col].nunique()
        print(f"  • {col}: {unique_times} unique time points")
else:
    print(f"\n⏰ TIME SERIES INFORMATION:")
    print("  • No time-related columns detected")

# Categorical Variables
if adata.obs is not None:
    categorical_cols = adata.obs.select_dtypes(include=['object', 'category']).columns
    non_time_categorical = [col for col in categorical_cols if col not in time_columns]
    if len(non_time_categorical) > 0:
        print(f"\n📋 CATEGORICAL VARIABLES:")
        for col in non_time_categorical:
            unique_vals = adata.obs[col].nunique()
            print(f"  • {col}: {unique_vals} unique values")
    else:
        print(f"\n📋 CATEGORICAL VARIABLES:")
        print("  • No non-time categorical variables found")

# Quality Assessment
print(f"\n✅ QUALITY ASSESSMENT:")
if hasattr(adata.X, 'data'):
    total_elements = adata.X.shape[0] * adata.X.shape[1]
    zero_elements = total_elements - adata.X.nnz
    print(f"  • Zero elements: {zero_elements:,} ({zero_elements/total_elements*100:.2f}%)")
    print(f"  • Data appears to be high-quality sparse single-cell data")
else:
    missing_count = np.sum(np.isnan(adata.X))
    zero_count = np.sum(adata.X == 0)
    print(f"  • Missing values: {missing_count:,}")
    print(f"  • Zero values: {zero_count:,} ({zero_count/adata.X.size*100:.2f}%)")

# Recommendations
print(f"\n💡 RECOMMENDATIONS FOR FURTHER ANALYSIS:")

if zero_genes > 0:
    print(f"  🔧 Data Cleaning:")
    print(f"    • Remove {zero_genes} genes with zero expression")

if low_expressed_genes > 0:
    print(f"    • Consider filtering {low_expressed_genes} genes expressed in < 1% of cells")

if outlier_cells > len(cell_counts) * 0.05:
    print(f"    • Investigate {outlier_cells} outlier cells (>5% of dataset)")

print(f"  📈 Normalization:")
print(f"    • Apply normalization (e.g., CPM, TPM, or scran)")
print(f"    • Consider log-transformation after normalization")

if time_columns:
    print(f"  ⏰ Time Series Analysis:")
    print(f"    • Perform time-specific differential expression analysis")
    print(f"    • Consider trajectory inference (e.g., Monocle, Slingshot)")
    print(f"    • Analyze temporal gene expression patterns")

if len(non_time_categorical) > 0:
    print(f"  📊 Group Analysis:")
    print(f"    • Perform differential expression by categorical variables")
    print(f"    • Consider batch effect correction if needed")

print(f"  🔬 Advanced Analysis:")
print(f"    • Perform clustering to identify cell types")
print(f"    • Apply dimensionality reduction (t-SNE, UMAP)")
print(f"    • Conduct pathway enrichment analysis")
print(f"    • Consider integration with other datasets if available")

print(f"\n🎯 NEXT STEPS:")
print(f"  1. Data preprocessing and quality control")
print(f"  2. Normalization and scaling")
print(f"  3. Feature selection")
print(f"  4. Dimensionality reduction and clustering")
print(f"  5. Cell type annotation")
print(f"  6. Downstream biological analysis")

print(f"\n✨ Analysis completed successfully! ✨")